# ANU ASTR4004 2026 - Week 4: Querying large datasets with ADQL


Author: Dr Sven Buder (sven.buder@anu.edu.au)

In [ ]:
try:
    %matplotlib inline
    %config InlineBackend.figure_format='retina'
except:
    pass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make the size and fonts larger for this presentation
plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 16
plt.rcParams['lines.linewidth'] = 2

from astropy.table import Table

In [ ]:
def query_gaia_archive_and_join_with_2mass():
    """
    Query the Gaia archive and join with 2MASS data.
    
    OUTPUT:
    A table with columns from both the Gaia and 2MASS catalogs, including object IDs, positions, and photometry.

    """

### Downloading Data via ADQL/SQL

Remember how we read in a data table from the GALAH survey earlier with spectroscopic measurements?

The table has the identifier of the 1.8 billion source large Gaia DR3 catalogue in it.

Downloading the full Gaia DR3 catalogue is not useful, but we can use the `astroquery` package to perform a query with the Astronomical Data Query Language (ADQL), an extension of the Structured Query Language (SQL) to include functions when querying data.

Let's try to download the matches in Gaia DR3!

In [ ]:
# If needed (and online), install dependencies:
# %pip install astroquery
from astroquery.gaia import Gaia

In [ ]:
# Extract only the Gaia DR3 source IDs for crossmatching
galah_data = Table.read('data/galah_dr3_allstar_m67_lite.csv')

# Some students may get a warning about integer overflow and that a column is reformatted into string.
# In that case, you may need to loop over each value and convert it individually.
galah_data['source_id'] = np.array([np.int64(x) for x in galah_data['dr3_source_id']])

gaia_source_ids = galah_data['source_id'].tolist()

# Convert the source IDs to an Astropy table to use in the query (to not upload too much data)
# with the actual identifier in Gaia DR3 (source_id)
gaia_source_ids_table = Table([gaia_source_ids], names=['source_id'])

# Define and execute the ADQL query to crossmatch with Gaia DR3
query = f"""
SELECT * 
FROM gaiadr3.gaia_source AS gaia
JOIN TAP_UPLOAD.t1 AS galah
ON gaia.source_id = galah.source_id
"""

# Upload the source_id table for crossmatching
job = Gaia.launch_job_async(query=query, upload_resource=gaia_source_ids_table, upload_table_name="t1")
gaiadr3_match = job.get_results()

In [ ]:
gaiadr3_match[:5]

### Joining 2 catalogues



In [ ]:
# fix missing keyword, because we need the same keyword *source_id* to join the 2 catalogues
galah_data['source_id'] = galah_data['dr3_source_id']

In [ ]:
from astropy.table import join

gaia_dr3_galah = join(gaiadr3_match, galah_data, keys='source_id')

In [ ]:
gaia_dr3_galah

In [ ]:
def plot_cmd_and_kiel(data, colormap_left = 'snr_c2_iraf', colormap_left_label = 'GALAH DR3 SNR CCD2'):

    # compare Gaia DR3 and GALAH DR3 measurements
    f, gs = plt.subplots(1, 2, figsize=(10,4))

    # Left panel (Color-Magnitude Diagram, CMD)
    ax = gs[0]
    ax.text(0.05, 0.95, 'a)', transform=ax.transAxes, fontsize=14, ha = 'left', va='top')

    sc = ax.scatter(
        data['bp_rp'],
        data['phot_g_mean_mag'] + 5 * np.log10(data['parallax']/100.), 
        c=data[colormap_left],
        cmap='viridis', s=10
    )
    ax.invert_yaxis()
    ax.set_xlabel(r'$G_\mathrm{BP} - G_\mathrm{RP}~/~\mathrm{mag}$')
    ax.set_ylabel(r'$G~/~\mathrm{mag}$')
    ax.set_title(r'$Gaia$ DR3')

    # Adding a colorbar for SNR in the left panel
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(colormap_left_label)

    # Right panel (Teff-logg diagram)
    ax = gs[1]
    ax.text(0.05, 0.95, 'b)', transform=ax.transAxes, fontsize=14, ha = 'left', va='top')
    sc2 = ax.scatter(
        data['teff'],
        data['logg'], 
        c=data['fe_h'],
        cmap='plasma', s=10
    )
    ax.invert_xaxis()
    ax.invert_yaxis()
    ax.set_xlabel(r'$T_\mathrm{eff}~/~\mathrm{K}$')
    ax.set_ylabel(r'$\log (g~/~\mathrm{cm\,s^{-2}}$')
    ax.set_title('GALAH DR3')

    # Adding a colorbar for [Fe/H] in the right panel
    cbar2 = plt.colorbar(sc2, ax=ax)
    cbar2.set_label('[Fe/H]')

    plt.tight_layout()
    plt.show()
    plt.close()
    
plot_cmd_and_kiel(data=gaia_dr3_galah)

### Cleaning Catalogues

Quite often, catalogues do not have all measurements.

Just above, you have seen the error message
```
RuntimeWarning: invalid value encountered in log10
gaia_dr3_galah['phot_g_mean_mag'] + 5 * np.log10(gaia_dr3_galah['parallax']/100.),
```

This message is expected if np.log10() is applied to a value that is not positive (this could be either negative value or value that is "Not a Value" aka NaN). Let's get a first idea:

In [ ]:
gaia_dr3_galah['parallax']/100.

We already see negative values, but we can also imagine values that are *Not a Number* aka NaN or they might have a bitmask "flag" that indicates their quality.

Negative parallax measurements are true measurements! They just tell us that the source is quite far away. Later in this course how we can use our **prior** knowledge that distances from us have to be positive to still extract something useful out of these measures.

For now, we will simply identify these measurements and not use them to avoid error messages.
For a research paper, this would be a selection cut that has needs to be documented for reproducability!

**NaN entries**  

You can identify NaN entries with the check, e.g. effective temperature $T_\mathrm{eff}$.
If you have checked if a value is NaN, you can also invert the result with a `~` (switch True <-> False):

In [ ]:
check_if_parallax_values_finite = np.isfinite(gaia_dr3_galah['parallax'])
check_if_parallax_values_nan = np.isnan(gaia_dr3_galah['parallax'])
check_if_parallax_values_not_fintie = ~np.isfinite(gaia_dr3_galah['parallax'])
check_if_parallax_positive = gaia_dr3_galah['parallax'] > 0

# you can also use the np.where function:
where_parallax_not_positive = np.where(~check_if_parallax_positive==True)

In [ ]:
gaia_dr3_galah['parallax'][where_parallax_not_positive]

**bitmask flags**

A bitmask is a way of storing multiple boolean (True/False) values in a single integer by representing each condition with a different bit in the binary representation of the number. For example, you might use bitmasks in a catalog to encode several flags in a compact form.

You can for example imagine a bitmask flag `0111`, which would add up to `8*0 + 4*1 + 2*1 + 1*1 = 7`. So with just 1 number, you can check for 4 different things!

You can also check if just a specific bitmask is raised:

In [ ]:
def is_bit_raised(flag, bit):
    """
    Check if *flag* has the value for a specific *bit* raised
    """
    return (flag & bit) != 0

In GALAH DR3, one of these quality flags is `flag_sp`, which includes a lot of details about quality checks and extra information about a star, for example, if we think it is a binary star.

According to the documentation (Table 4 from https://ui.adsabs.harvard.edu/abs/2021MNRAS.506..150B)
that would mean raised flags 32 (spectroscopic binary) or 64 (photometric binary).

We can find the stars that are flagged as binaries with an `|` check, which means `or`.
You could also find stars that are spectroscopic *and* photometric binaries by replacing `|` with `&`:

In [ ]:
check_if_binary = (
    is_bit_raised(gaia_dr3_galah['flag_sp'], 32) &
    is_bit_raised(gaia_dr3_galah['flag_sp'], 64)
)

gaia_dr3_galah['binary'] = check_if_binary

In [ ]:
plot_cmd_and_kiel(
    gaia_dr3_galah[check_if_parallax_positive],
    colormap_left='binary',
    colormap_left_label='Binary True/False'
)

# Another Solved Example

**Goal:** Retrieve bright stars within 1° of (RA=180°, Dec=0°) and compute absolute G.

In [ ]:
adql = """
SELECT TOP 200
  source_id, ra, dec,
  phot_g_mean_mag,
  bp_rp,
  parallax,
  (phot_g_mean_mag + 5*LOG10(parallax/1000.0) + 5) AS abs_g
FROM gaiadr3.gaia_source
WHERE phot_g_mean_mag < 15
  AND parallax > 0
  AND CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', 180.0, 0.0, 1.0)
      ) = 1;
"""
job = Gaia.launch_job_async(adql)
demo = job.get_results()
display(demo.to_pandas().head())
print(adql)


### Plotting templates (with units in labels)
Use these once you have a result table with the appropriate columns.


In [ ]:
# Example plotting function for a Gaia-style CMD (abs G vs. BP-RP)
def plot_cmd(table, color_col='bp_rp', abs_g_col='abs_g', title='Gaia CMD'):
    df = table if isinstance(table, pd.DataFrame) else table.to_pandas()
    x = df[color_col].values
    y = df[abs_g_col].values
    plt.figure(figsize=(6,5))
    plt.scatter(x, y, s=4, alpha=0.6)
    plt.gca().invert_yaxis()
    plt.xlabel(r'$G_\mathrm{BP} − G_\mathrm{RP}~/~\mathrm{mag}$')
    plt.ylabel(r'$M_G~/~\mathrm{mag}$')
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Proper motion vector-point diagram
def plot_pm(table, pmra_col='pmra', pmdec_col='pmdec', title='Proper motions'):
    df = table if isinstance(table, pd.DataFrame) else table.to_pandas()
    x = df[pmra_col].values
    y = df[pmdec_col].values
    plt.figure(figsize=(6,5))
    plt.scatter(x, y, s=4, alpha=0.6)
    plt.xlabel(r'$\mu_\mathrm{RA}~/~\mathrm{mas\,yr^{-1}}$')
    plt.ylabel(r'$\mu_\mathrm{Dec}~/~\mathrm{mas\,yr^{-1}}$')
    plt.title(title)
    plt.axhline(0, lw=0.8)
    plt.axvline(0, lw=0.8)
    plt.tight_layout()
    plt.show()


## Exercise 1 (5 min) — Color–magnitude cut with absolute G

**Task:** Within 2° of (RA=150°, Dec=2°), select up to 500 stars with G<16 and 0.5 ≤ BP−RP ≤ 1.5. Compute absolute G.
Write the ADQL in the cell below and (if online) execute it.

You can find the solution at the end of this notebook.

In [ ]:
# TODO: Write your ADQL as a string in `adql` and run with astroquery if online.
adql = """
-- your ADQL here
"""
# job = Gaia.launch_job_async(adql)
# ex1 = job.get_results()
# display(ex1.to_pandas().head())
# plot_cmd(ex1, color_col='bp_rp', abs_g_col='abs_g', title='Exercise 1: CMD')


## Exercise 2 (7–8 min) — White dwarf candidates (quality + CMD)
**Task:** Find up to 500 **white dwarf candidates** within 100 pc using Gaia DR3. Apply:
- parallax > 10 mas; RUWE < 1.4; visibility_periods_used ≥ 8
- S/N gates: photometric flux_over_error thresholds (suggested G>50, BP/RP>20)
- BP/RP excess factor within a color-dependent band
- Color sanity: −0.5 ≤ BP−RP ≤ 1.5
- CMD cuts: 10 ≤ M_G ≤ 16 and a sloped cut `M_G > 5*BP−RP + 8.5`


In [ ]:
# TODO: Write ADQL in `adql` and run (if online). Then plot CMD and proper motions.
adql = """
-- your ADQL here
"""
# job = Gaia.launch_job_async(adql)
# ex2 = job.get_results()
# df2 = ex2.to_pandas()
# plot_cmd(df2, color_col='bp_rp', abs_g_col='abs_g', title='Exercise 2: WD candidates')
# plot_pm(df2, pmra_col='pmra', pmdec_col='pmdec', title='Exercise 2: Proper motions')


## Exercise 3 — Join Gaia with another catalogue: Gaia ↔ 2MASS PSC (archive neighbor table)
This is an adjusted example from the Gaia Archive (but also requesting gaia.phot_g_mean_mag and gaia.bp_rp)

In [ ]:
adql_gaia_only = """
SELECT gaia.source_id, gaia.phot_g_mean_mag, gaia.bp_rp, gaia.parallax, (gaia.phot_g_mean_mag + 5*LOG10(gaia.parallax/1000.0) + 5) AS abs_g
FROM gaiadr3.gaia_source AS gaia
WHERE
gaia.parallax > 50
"""
job = Gaia.launch_job_async(adql_gaia_only)
ex3 = job.get_results()
df3 = ex3.to_pandas()

plot_cmd(df3, color_col='bp_rp', abs_g_col='abs_g', title='Gaia CMD')


In [ ]:
adql_join_2mass = """
SELECT gaia.source_id, gaia.phot_g_mean_mag, gaia.bp_rp, gaia.parallax, tmass.j_m, tmass.ks_m
FROM gaiadr3.gaia_source AS gaia
JOIN gaiadr3.tmass_psc_xsc_best_neighbour AS xmatch USING (source_id)
JOIN gaiadr3.tmass_psc_xsc_join AS xjoin USING (clean_tmass_psc_xsc_oid)
JOIN gaiadr1.tmass_original_valid AS tmass ON
   xjoin.original_psc_source_id = tmass.designation
WHERE
gaia.parallax > 50 AND
tmass.ph_qual = 'AAA'
"""
job = Gaia.launch_job_async(adql_join_2mass)
ex3 = job.get_results()
df3 = ex3.to_pandas()

In [ ]:
# Convert apparent to absolute magnitude

# your code here...

# Plot Gaia CMD -- Note: we have not queried abs_g yet!
# plot_cmd(df3, color_col='bp_rp', abs_g_col='abs_g', title='Gaia CMD')

# Convert apparent to absolute magnitude for 2MASS

# your code here...

# Calculate 2MASS color


# your code here...

#plot_cmd(df3, color_col='j_ks', abs_g_col='abs_k', title='2MASS CMD')

# Exercise 4: Join with any other catalogue in the archive -- here: Bailer-Jones et al. (2021) distances

You can join a lot of different catalogues both on the Gaia archive or via other TOPCAT.

In the case of distances, you will have to figure out what distance a certain parallax value corresponds to and then use that as <YOUR_DISTANCE_LIMIT>

In [ ]:
adql_join_bj21 = """
-- Replace <bj_table> with the BJ21 table at your TAP service (e.g., external.gaiaedr3_distance)
SELECT TOP 1000
  g.source_id, g.ra, g.dec,
  g.phot_g_mean_mag,
  bj.r_med_photogeo AS dist_pc
FROM gaiadr3.gaia_source AS g
JOIN <bj_table> AS bj
  ON g.source_id = bj.source_id
WHERE bj.r_med_photogeo < <YOUR_DISTANCE_LIMIT>;
"""
# job = Gaia.launch_job_async(adql_join_bj21)
# ex4 = job.get_results()
# df4 = ex4.to_pandas()

# If you are interested: Visualising targets with ipyaladin

You can find ipyaladin at https://github.com/cds-astro/ipyaladin

In [ ]:
from ipyaladin import Aladin
aladin = Aladin()
aladin

# Initialize Aladin and set the view to M67 in 2MASS J color image
aladin = Aladin(target='M67', survey='2MASS-J', fov=0.5)  # Adjust the field of view as needed

# Display Aladin widget
aladin

## Cheet Sheets for Exercises

### Cheat Sheet 1 — Basic SELECT & Geometry
```sql
SELECT TOP N columns
FROM schema.table
WHERE <filter1>
  AND <filter2>
  AND CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', ra0, dec0, radius_deg)
      ) = 1;
```
**Absolute G:** `(phot_g_mean_mag + 5*LOG10(parallax/1000.0) + 5)`  
**Color sanity:** `bp_rp BETWEEN -0.5 AND 4.0`


### Cheat Sheet 2 — Quality Cuts & CMD Regions
```
ruwe < 1.4
visibility_periods_used >= 8
parallax_over_error > X
phot_*_mean_flux_over_error > Y
bp_rp BETWEEN low AND high

phot_bp_rp_excess_factor BETWEEN
   (1.0 + 0.015*POWER(bp_rp,2)) AND (1.3 + 0.060*POWER(bp_rp,2))

-- Absolute magnitude range and sloped separators
abs_g BETWEEN min AND max
abs_g > slope * bp_rp + intercept
```


### Cheat Sheet 3 — Joins
**Gaia ↔ 2MASS (neighbor table):**
```sql
SELECT g.source_id, t.j_m
FROM gaiadr3.gaia_source AS g
JOIN gaiadr3.tmass_psc_xsc_best_neighbour AS x ON g.source_id = x.source_id
JOIN gaiadr3.tmass_psc AS t ON x.tmass_psc_id = t.tmass_psc_id;
```
**Gaia ↔ Bailer-Jones 2021 (table name varies by TAP):**
```sql
SELECT g.source_id, bj.r_med_photogeo AS dist_pc
FROM gaiadr3.gaia_source AS g
JOIN <bj_table> AS bj ON g.source_id = bj.source_id;
```


# Solutions for Exercises

## Exercise 1

In [ ]:
adql_solution_1 = """
SELECT TOP 500
  source_id, ra, dec,
  phot_g_mean_mag, bp_rp, parallax,
  (phot_g_mean_mag + 5*LOG10(parallax/1000.0) + 5) AS abs_g
FROM gaiadr3.gaia_source
WHERE phot_g_mean_mag < 16
  AND bp_rp BETWEEN 0.5 AND 1.5
  AND parallax > 0
  AND CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', 150.0, 2.0, 2.0)
      ) = 1;
"""


## Exercise 2

In [ ]:
adql_solution_2 = """
SELECT TOP 500
  g.source_id, g.ra, g.dec,
  g.phot_g_mean_mag AS g_mag,
  g.bp_rp,
  g.parallax,
  g.pmra, g.pmdec,
  (g.phot_g_mean_mag + 5*LOG10(g.parallax/1000.0) + 5) AS abs_g,
  SQRT(POWER(g.pmra,2) + POWER(g.pmdec,2)) AS mu,
  4.74047*SQRT(POWER(g.pmra,2) + POWER(g.pmdec,2))/g.parallax AS v_tan_kms,
  g.ruwe, g.phot_bp_rp_excess_factor, g.visibility_periods_used
FROM gaiadr3.gaia_source AS g
WHERE g.parallax > 10 AND g.parallax_over_error > 10
  AND g.ruwe < 1.4
  AND g.visibility_periods_used >= 8
  AND g.phot_g_mean_flux_over_error > 50
  AND g.phot_bp_mean_flux_over_error > 20
  AND g.phot_rp_mean_flux_over_error > 20
  AND g.bp_rp BETWEEN -0.5 AND 1.5
  AND g.phot_bp_rp_excess_factor > (1.0 + 0.015*POWER(g.bp_rp,2))
  AND g.phot_bp_rp_excess_factor < (1.3 + 0.060*POWER(g.bp_rp,2))
  AND (g.phot_g_mean_mag + 5*LOG10(g.parallax/1000.0) + 5) BETWEEN 10 AND 16
  AND (g.phot_g_mean_mag + 5*LOG10(g.parallax/1000.0) + 5) > (5.0*g.bp_rp + 8.5);
"""

## Exercise 3a

In [ ]:
adql_join_2mass = """
SELECT gaia.source_id, gaia.phot_g_mean_mag, gaia.bp_rp, gaia.parallax, tmass.j_m, tmass.ks_m
FROM gaiadr3.gaia_source AS gaia
JOIN gaiadr3.tmass_psc_xsc_best_neighbour AS xmatch USING (source_id)
JOIN gaiadr3.tmass_psc_xsc_join AS xjoin USING (clean_tmass_psc_xsc_oid)
JOIN gaiadr1.tmass_original_valid AS tmass ON
   xjoin.original_psc_source_id = tmass.designation
WHERE
gaia.parallax > 50 AND
tmass.ph_qual = 'AAA'
"""
job = Gaia.launch_job_async(adql_join_2mass)
ex3 = job.get_results()
df3 = ex3.to_pandas()

In [ ]:
# Convert apparent to absolute magnitude
df3['abs_g'] = df3['phot_g_mean_mag'] + 5 * np.log10(df3['parallax']/1000.0) + 5

# Plot Gaia CMD
plot_cmd(df3, color_col='bp_rp', abs_g_col='abs_g', title='Gaia CMD')

# Convert apparent to absolute magnitude for 2MASS
df3['abs_k'] = df3['ks_m'] + 5 * np.log10(df3['parallax']/1000.0) + 5

# Calculate 2MASS color
df3['j_ks'] = df3['j_m'] - df3['ks_m']

plot_cmd(df3, color_col='j_ks', abs_g_col='abs_k', title='2MASS CMD')

In [ ]:
adql_join_bj21 = """
-- Replace <bj_table> with the BJ21 table at your TAP service (e.g., external.gaiaedr3_distance)
SELECT
  g.source_id, g.ra, g.dec,
  g.phot_g_mean_mag,
  bj.r_med_photogeo AS dist_pc
FROM gaiadr3.gaia_source AS g
JOIN external.gaiaedr3_distance AS bj
  ON g.source_id = bj.source_id
WHERE bj.r_med_photogeo < 20;
"""
job = Gaia.launch_job_async(adql_join_bj21)
ex4 = job.get_results()
df4 = ex4.to_pandas()